In [0]:
from pyspark.sql.functions import col, to_date, year, month, when, lower, trim, upper, datediff, when

In [0]:
%sql
create schema if not exists 02_silver_catalog.silver 

In [0]:
bronze_schema = "01_bronze_catalog.bronze_azure_blob."
silver_schema = "02_silver_catalog.silver."

### Customer Survey Table

In [0]:
df_customer_survey = spark.table(bronze_schema+"customer_survey")
df_customer_survey = df_customer_survey.drop("_line","_file","_fivetran_synced","_modified","version_no")

df_customer_survey.display()


In [0]:
customer_survey = df_customer_survey.withColumns({

    # Boolean → int
    "responded_flag": col("responded_flag").cast("int"),

    # Preserve timestamps
    "survey_sent_ts": col("survey_sent_date"),
    "survey_response_ts": col("survey_response_date"),

    # Add date columns
    "survey_sent_date": to_date(col("survey_sent_date")),
    "survey_response_date": to_date(col("survey_response_date")),

    # Ratings cleanup
    "cleanliness_rating": col("cleanliness_rating").cast("double"),
    "communication_rating": col("communication_rating").cast("double"),
    "work_quality_rating": col("work_quality_rating").cast("double"),
    "delivered_on_time_rating": col("delivered_on_time_rating").cast("double"),
    "overall_satisfaction_rating": col("overall_satisfaction_rating").cast("double")
})

display(customer_survey)

In [0]:
customer_survey.write.format("delta").option("mergeSchema","true").mode("overwrite").saveAsTable(silver_schema+"customer_survey")

### Estimate Table


In [0]:
df_estimate = spark.table(bronze_schema+"estimate")
df_estimate = df_estimate.drop("_line","_file","_fivetran_synced","_modified","version_no")
df_estimate.display()

In [0]:

estimate = df_estimate.withColumns({

    # Preserve timestamp
    "created_ts": col("created_at"),

    # Add date
    "created_date": to_date(col("created_at")),

    # Clean text
    "estimator_name": trim(col("estimator_name")),
    "estimate_type": lower(trim(col("estimate_type"))),

    # Amount
    "estimate_amount": col("estimate_amount").cast("double")
})

display(estimate)

In [0]:
estimate.write.format("delta").option("mergeSchema","true").mode("overwrite").saveAsTable(silver_schema+"estimate")

### Invoice Table

In [0]:
df_invoice = spark.table(bronze_schema+"invoice")
invoice = df_invoice.drop("_line","_file","_fivetran_synced","_modified")
invoice.display()

In [0]:
invoice = invoice.withColumns({

    # Preserve timestamp
    "invoice_ts": col("invoice_date"),

    # Add date
    "invoice_date": to_date(col("invoice_date")),

    # Clean text
    "payment_mode": lower(trim(col("payment_mode"))),
    "currency": upper(trim(col("currency"))),

    # Amount
    "invoice_amount": col("invoice_amount").cast("double")
})

display(invoice)

In [0]:
invoice.write.format("delta").mode("overwrite").saveAsTable(silver_schema+"invoice")

### NS Budget Table

In [0]:
df_ns_budget = spark.table(bronze_schema+"ns_budget")
ns_budget = df_ns_budget.drop("_line","_file","_fivetran_synced","_modified")
ns_budget.display()

In [0]:

ns_budget = ns_budget.withColumns({

    # Rename + convert
    "budget_date": to_date(col("month")),

    # Derived columns
    "budget_year": year(to_date(col("month"))),
    "budget_month": month(to_date(col("month"))),

    # Amount
    "budget_amount": col("budget_amount").cast("double")
}).drop("month")


display(ns_budget)

In [0]:
ns_budget.write.format("delta").mode("overwrite").saveAsTable(silver_schema+"ns_budget")

### Store table

In [0]:
df_store = spark.table(bronze_schema+"store")
df_store = df_store.drop("_line","_file","_fivetran_synced","_modified")
df_store.display()


In [0]:
store = df_store.withColumns({

    # Clean text
    "store_name": trim(col("store_name")),
    "city": lower(trim(col("city"))),
    "state": lower(trim(col("state"))),
    "manager_name": trim(col("manager_name")),
    "store_type": lower(trim(col("store_type"))),

    # Types
    "opened_year": col("opened_year").cast("int")
})

store.display()

In [0]:
store.write.format("delta").mode("overwrite").saveAsTable(silver_schema+"store")

### Order Table

In [0]:
df_order = spark.table(bronze_schema+"order")
df_order = df_order.drop("_line","_file","_fivetran_synced","_modified")
df_order.display()


In [0]:
order = df_order.withColumns({

    # Clean text
    "service_type": lower(trim(col("service_type"))),
    "vehicle_make": lower(trim(col("vehicle_make"))),
    "vehicle_model": lower(trim(col("vehicle_model"))),
    "customer_name": trim(col("customer_name")),
    "technician_name": trim(col("technician_name")),
    "order_status": lower(trim(col("order_status"))),

    # Add date columns
    "vehicle_in_date": to_date(col("vehicle_in_datetime")),
    "vehicle_out_date": to_date(col("vehicle_out_datetime")),
    "work_start_date": to_date(col("actual_work_start_datetime")),
    "completion_date": to_date(col("actual_completion_datetime")),
    "delivery_date": to_date(col("actual_delivery_datetime")),

    # Derived metrics (NULL-safe + controlled)
    "days_in_shop": when(
        col("actual_delivery_datetime").isNotNull() & col("vehicle_in_datetime").isNotNull(),
        datediff(col("actual_delivery_datetime"), col("vehicle_in_datetime"))
    ),

    "work_start_delay_days": when(
        col("actual_work_start_datetime").isNotNull() & col("planned_work_start_datetime").isNotNull(),
        datediff(col("actual_work_start_datetime"), col("planned_work_start_datetime"))
    ),

    "completion_delay_days": when(
        col("actual_completion_datetime").isNotNull() & col("planned_completion_datetime").isNotNull(),
        datediff(col("actual_completion_datetime"), col("planned_completion_datetime"))
    ),

    "delivery_delay_days": when(
        col("actual_delivery_datetime").isNotNull() & col("promised_delivery_datetime").isNotNull(),
        datediff(col("actual_delivery_datetime"), col("promised_delivery_datetime"))
    )
})

order.display()

In [0]:
order.write.format("delta").mode("overwrite").saveAsTable(silver_schema+"order")